<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">La Clasificación de Texto con RNN y LSTM</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Este notebook lleva los modelos recurrentes a una tarea concreta: clasificar texto. Aquí se ve cómo la representación secuencial se conecta con una salida supervisada útil.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Pasar de la teoría de secuencias a una tarea aplicada ayuda a entender qué aprenden realmente estas arquitecturas.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A interpretar cómo una RNN o una LSTM pueden convertir una secuencia textual en una decisión de clasificación.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">En NLP aplicado no basta con procesar una secuencia; hay que traducir esa información a una decisión final coherente con la tarea.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Clasificar texto implica resumir información relevante de la secuencia para asignarla a una clase o etiqueta.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Definición del corpus y etiquetas
"""Este bloque define un corpus simple y sus etiquetas asociadas.
La variable "corpus" contiene frases que serán clasificadas con los modelos recurrentes RNN y LSTM.
La variable "etiquetas" asigna una categoría numérica a cada frase.
Estas estructuras podrían utilizarse para entrenar un modelo, pero este ejemplo solo realiza pasadas hacia delante y no utiliza las etiquetas para entrenar."""

corpus = [
    "este producto es excelente y funciona muy bien",
    "la experiencia fue terrible y no lo recomiendo",
    "me encanta la calidad y el rendimiento",
    "el servicio fue muy malo y decepcionante"
]

etiquetas = [1, 0, 1, 0]

corpus, etiquetas # Muestra el corpus y las etiquetas asociadas para cada frase.


In [ ]:
# Bloque 2 — Tokenización y construcción del vocabulario
"""Este bloque tokeniza el corpus y construye un vocabulario.
Utilizamos una comprensión de lista "[]" para convertir cada frase del corpus en una lista de tokens (palabras individuales).
La subexpresión "for frase in corpus" recorre cada frase en el corpus original.
La subexpresión "frase.split()" divide cada frase en una lista de palabras (tokens).
Esta comprensión de lista permite tokenizar todo el corpus de manera concisa y eficiente.
La variable "corpus_tokenizado" almacena el corpus segmentado en una lista de listas de tokens por frase.

Aplicamos el constructor "set()" a la expresión "(token for frase in corpus_tokenizado for token in frase)" para obtener un conjunto de tokens únicos.
La subexpresión "for frase in corpus_tokenizado" recorre las frases del corpus tokenizado.
La subexpresión "for token in frase" recorre cada token dentro de la frase actual.
Aplicamos el constructor "list()" al conjunto resultante de tokens únicos para obtener una lista de tokens sin duplicados.
La variable "tokens_unicos" contiene la lista de tokens sin duplicados.

Utilizamos una comprensión de diccionario "{}" para asignar un índice único a cada token en la lista de tokens sin duplicados.
La subexpresión "token: idx" asigna a cada token su índice en la lista de tokens sin duplicados.
La subexpresión "for idx, token in enumerate(tokens_unicos)" recorre cada token junto con su índice en la lista de tokens sin duplicados.
La clase "enumerate()" enumera los elementos y proporciona un índice para cada elemento en la secuencia.
La variable "vocabulario" contiene el diccionario que mapea cada token único a su índice correspondiente.
El orden del conjunto no está garantizado entre procesos, por lo que las asignaciones de índices pueden variar."""

corpus_tokenizado = [frase.split() for frase in corpus]

tokens_unicos = list(set(token for frase in corpus_tokenizado for token in frase))

vocabulario = {token: idx for idx, token in enumerate(tokens_unicos)}

corpus_tokenizado, tokens_unicos, vocabulario # Muestra el corpus tokenizado, los tokens únicos y el vocabulario generado.


In [ ]:
# Bloque 3 — Conversión del corpus a secuencias de índices
"""Este bloque convierte cada frase tokenizada en una secuencia de índices.
Utilizamos una comprensión de lista "[]" para convertir cada token en su índice correspondiente en el vocabulario.
La subexpresión "vocabulario[token]" obtiene el índice correspondiente a cada token en el vocabulario definido previamente.
La subexpresión "for token in frase" recorre cada token en la lista de tokens del texto original.
La subexpresión "for frase in corpus_tokenizado" recorre cada frase tokenizada en el corpus.
Esta comprensión de lista permite transformar el texto tokenizado en una secuencia de índices de manera eficiente.

La variable "secuencias" almacena la secuencia de índices correspondiente a cada frase tokenizada.
Estas secuencias serán procesadas por modelos recurrentes para clasificación."""

secuencias = [[vocabulario[token] for token in frase] for frase in corpus_tokenizado]

secuencias # Muestra las secuencias de índices correspondientes a cada frase tokenizada del corpus.

In [ ]:
# Bloque 4 — Padding para longitud fija
"""Este bloque aplica truncado y padding para obtener secuencias de longitud uniforme.
La variable "longitud_objetivo" define la longitud deseada para cada secuencia, en este caso, 10 elementos.

Utilizamos una comprensión de lista para aplicar el padding a cada secuencia en la lista "secuencias".
La subexpresión "secuencia[:longitud_objetivo]" toma los primeros 10 elementos de la secuencia.
La subexpresión [0] * (longitud_objetivo - len(secuencia)) genera una lista de ceros para rellenar la secuencia hasta la longitud deseada.
La subexpresión "for secuencia in secuencias" itera sobre cada secuencia en la lista "secuencias".

El operador aritmético "+" concatena la sublista de los primeros 10 elementos con la lista de ceros generada para el padding.
La variable "secuencias_padded" almacena las secuencias finales con longitud uniforme.
El índice 0 también corresponde a un token del vocabulario, por lo que el padding no se distingue de ese token.
Los modelos procesarán los ceros añadidos como pasos de la secuencia, sin máscara ni exclusión del padding."""

longitud_objetivo = 10

secuencias_padded = [secuencia[:longitud_objetivo] + [0] * (longitud_objetivo - len(secuencia)) for secuencia in secuencias]

secuencias_padded # Muestra las secuencias ajustadas con padding para una longitud uniforme.


In [ ]:
# Bloque 5 — Conversión a tensores
"""Este bloque convierte las secuencias y etiquetas en tensores de PyTorch.
Este bloque importa "torch" para trabajar con tensores en PyTorch.
Aplicamos la función "torch.tensor()" a las variables "secuencias_padded" y "etiquetas", respectivamente, para obtener tensores de PyTorch.
Las variables "X" y "y" almacenan los tensores de entrada y etiquetas, respectivamente, obtenidos a partir de las secuencias y etiquetas originales.
El ejemplo utiliza "X" para la pasada hacia delante, pero no utiliza "y" para entrenar ni evaluar las predicciones."""

import torch

X = torch.tensor(secuencias_padded)
y = torch.tensor(etiquetas)

X, y # Muestra los tensores de entrada y etiquetas, respectivamente.


In [ ]:
# Bloque 6 — Definición de la capa de embeddings
"""Este bloque define una capa de embeddings que transforma índices en vectores densos.
Este bloque importa "torch.nn" para definir modelos de redes neuronales recurrentes.
La variable "embedding_dim" se establece en 16, lo que significa que cada índice se representará en un vector denso de 16 dimensiones.

Aplicamos la clase "nn.Embedding()" a la variable "capa_embeddings" para crear la capa de embeddings que transformará índices en vectores densos.
El parámetro "num_embeddings=" indica el tamaño del vocabulario, en este caso, la longitud del diccionario obtenida por "len(vocabulario)".
El parámetro "embedding_dim=" indica la dimensionalidad de los vectores densos, en este caso, el valor de "embedding_dim", es decir, 16.
La variable "capa_embeddings" almacena una instancia de la clase "nn.Embedding()" que se utilizará para convertir índices de palabras en vectores densos.

Asignamos la llamada a la instancia de la clase a la variable "X_embedded" mediante la expresión "capa_embeddings(X)".
El argumento "X" representa el tensor de secuencias de índices de palabras que se desea convertir en vectores densos.
La variable "X_embedded" contiene las secuencias convertidas en vectores, incluidos los correspondientes al índice 0 usado como padding.
La capa está inicializada sin entrenamiento y no configura un índice especial de padding."""

import torch.nn as nn

embedding_dim = 16

capa_embeddings = nn.Embedding(num_embeddings=len(vocabulario), embedding_dim=embedding_dim)

X_embedded = capa_embeddings(X)

X_embedded # Muestra las secuencias convertidas en vectores densos.


In [ ]:
# Bloque 7 — Definición de una RNN para clasificación
"""Este bloque define una RNN simple para clasificación de texto.
La variable "hidden_size" define la dimensionalidad del estado oculto, en este caso, 12.

Aplicamos la clase "nn.RNN()" a la variable "rnn" para definir una red neuronal recurrente simple.
El parámetro "input_size=" indica la dimensionalidad de los embeddings de entrada, en este caso, 16 dimensiones.
El parámetro "hidden_size=" define la dimensionalidad del estado oculto, en este caso, 12 dimensiones.
El parámetro "batch_first=" indica si la primera dimensión de la entrada corresponde al tamaño del batch; en este caso, se establece como "True".
La variable "rnn" almacena una instancia del modelo recurrente que procesará la secuencia de entrada.

Aplicamos la clase "nn.Linear()" a la variable "capa_salida" para definir la capa lineal que produce la predicción final.
El argumento "hidden_size" indica la dimensionalidad de la entrada, en este caso, 12 dimensiones del estado oculto.
El argumento "2" indica la dimensionalidad de la salida, en este caso, 2 dimensiones.
La variable "capa_salida" almacena la instancia de la capa lineal que se utilizará para generar los logits de ambas arquitecturas."""

hidden_size = 12

rnn = nn.RNN(input_size=embedding_dim, hidden_size=hidden_size, batch_first=True)

capa_salida = nn.Linear(hidden_size, 2)

rnn, capa_salida # Muestra las características de la RNN y la capa lineal de salida.


In [ ]:
# Bloque 8 — Forward pass con RNN
"""Este bloque realiza el procesamiento de las secuencias mediante la RNN.
Aplicamos la función "torch.zeros()" a la variable "estado_inicial" para inicializar el estado oculto con ceros en todas sus dimensiones.
El primer argumento "1" indica el número de capas y direcciones de la RNN en este caso.
El segundo argumento "X.size(0)" indica el tamaño del batch, es decir, el número de secuencias que se procesarán simultáneamente.
El método ".size(0)" aplicado a "X" devuelve el tamaño de su primera dimensión.
El argumento "hidden_size" indica la dimensionalidad del estado oculto, en este caso, 12.
La variable "estado_inicial" almacena el estado oculto inicial de la RNN, que en este caso es un tensor de ceros en todas sus dimensiones.

Asignamos de forma múltiple el resultado de la llamada a la instancia de la clase a las variables "salida_rnn" y "estado_final_rnn" mediante la expresión "rnn(X_embedded, estado_inicial)".
El argumento "X_embedded" es el tensor de entrada que ha sido previamente transformado mediante la capa de embeddings.
El argumento "estado_inicial" es el estado oculto de partida definido previamente para la RNN.

La variable "salida_rnn" contiene las salidas generadas en cada paso de la secuencia en forma de tensor.
La variable "estado_final_rnn" contiene el estado oculto final de la RNN después de procesar toda la secuencia.

Asignamos la llamada a la instancia de la clase a la variable "logits" mediante la expresión "capa_salida(estado_final_rnn.squeeze(0))".
El argumento "estado_final_rnn" representa el estado oculto final de la RNN después de procesar toda la secuencia.
Acompañamos el argumento con el método ".squeeze(0)" para eliminar la dimensión de capa, cuyo tamaño es 1.
La variable "logits" almacena las puntuaciones de clase antes de aplicar softmax; los estados finales incluyen los pasos de padding."""

estado_inicial = torch.zeros(1, X.size(0), hidden_size)

salida_rnn, estado_final_rnn = rnn(X_embedded, estado_inicial)

logits = capa_salida(estado_final_rnn.squeeze(0))

salida_rnn, estado_final_rnn, logits # Muestra las salidas de la RNN, el estado final y los logits.


In [ ]:
# Bloque 9 — Definición de una LSTM para clasificación
"""Este bloque define una LSTM para clasificación de texto.
Aplicamos la clase "nn.LSTM()" a la variable "lstm" para definir una red neuronal recurrente con memoria mejorada.
El parámetro "input_size=" indica la dimensionalidad de los embeddings de entrada, en este caso, 16 dimensiones.
El parámetro "hidden_size=" define la dimensionalidad del estado oculto, en este caso, 12 dimensiones.
El parámetro "batch_first=" indica si la primera dimensión de la entrada corresponde al tamaño del batch; en este caso, se establece como "True".

La variable "lstm" almacena una arquitectura recurrente diseñada para facilitar el aprendizaje de dependencias a largo plazo.
La LSTM incorpora mecanismos de puertas que controlan el flujo de información."""

lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_size, batch_first=True)

lstm # Muestra las características de la LSTM definida.

In [ ]:
# Bloque 10 — Forward pass con LSTM
"""Este bloque procesa las secuencias mediante la LSTM.
Aplicamos la función "torch.zeros()" a las variables "estado_oculto" y "estado_celda", respectivamente, para inicializar el estado oculto 
y el estado de celda con ceros en todas sus dimensiones.
        
El primer argumento "1" en cada caso indica el número de capas y direcciones de la LSTM en este ejemplo.
El argumento "X.size(0)" en cada caso indica el tamaño del batch, es decir, el número de secuencias que se procesarán simultáneamente.
El método ".size(0)" aplicado a "X" devuelve el tamaño de su primera dimensión.
El argumento "hidden_size" en cada caso indica la dimensionalidad del estado oculto, en este caso, 12.
Las variables "estado_oculto" y "estado_celda" almacenan el estado oculto inicial y el estado de celda inicial de la LSTM, 
que en este caso son tensores de ceros en todas sus dimensiones.

Asignamos de forma múltiple el resultado de la llamada a la instancia de la clase "LSTM" a las variables "salida_lstm" y "estados_finales_lstm" mediante 
la expresión "lstm(X_embedded, (estado_oculto, estado_celda))".
El argumento "X_embedded" es el conjunto de tensores densos de entrada que queremos procesar mediante la LSTM.
El argumento "estado_oculto" es el estado oculto de partida definido previamente para la LSTM.
El argumento "estado_celda" es el estado de celda de partida definido previamente para la LSTM.
En este caso, se pasan los estados en forma de tupla "()", donde el primer elemento es el estado oculto y el segundo elemento es el estado de celda.

La variable "salida_lstm" almacena las representaciones generadas en cada paso.
La variable "estados_finales_lstm" almacena los estados oculto y de celda resultantes de la LSTM después de procesar toda la secuencia.

Asignamos la llamada a la instancia de la clase "capa_salida" a la variable "logits_lstm" mediante la expresión "capa_salida(estados_finales_lstm[0].squeeze(0))".
El argumento "estados_finales_lstm[0]" representa el estado oculto final de la LSTM después de procesar toda la secuencia.
Acompañamos el argumento con el método ".squeeze(0)" para eliminar la dimensión de capa, cuyo tamaño es 1.
La variable "logits_lstm" almacena las puntuaciones de clase antes de aplicar softmax; los estados finales incluyen los pasos de padding."""

estado_oculto = torch.zeros(1, X.size(0), hidden_size)
estado_celda = torch.zeros(1, X.size(0), hidden_size)

salida_lstm, estados_finales_lstm = lstm(X_embedded, (estado_oculto, estado_celda))

logits_lstm = capa_salida(estados_finales_lstm[0].squeeze(0))

salida_lstm, estados_finales_lstm, logits_lstm # Muestra las salidas de la LSTM, los estados finales y los logits.


In [ ]:
# Bloque 11 — Resultados finales
"""Este bloque organiza los resultados del proceso de clasificación mediante RNN y LSTM.
La variable "resultado" almacena en forma de diccionario los logits y las predicciones.
Las predicciones proceden de modelos sin entrenar y no permiten comparar el rendimiento de las arquitecturas."""

resultado = {
    "logits_rnn": logits,
    "logits_lstm": logits_lstm,
    "predicciones_rnn": torch.argmax(logits, dim=1),       # La función "torch.argmax" obtiene la clase con mayor logit para cada secuencia.
    "predicciones_lstm": torch.argmax(logits_lstm, dim=1)  # La función "torch.argmax" obtiene la clase con mayor logit para cada secuencia.
}

resultado # Muestra el diccionario con los resultados finales de la clasificación mediante RNN y LSTM.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto una aplicación supervisada concreta de modelos secuenciales sobre texto.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">La red debe extraer contexto relevante de la secuencia para convertirlo en una predicción útil.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Centrarte solo en la arquitectura y descuidar la relación entre representación textual, tarea y criterio de evaluación.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en las Limitaciones de los Modelos Recurrentes para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en las Limitaciones de los Modelos Recurrentes y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">03_Modelos_recurrentes_para_texto/04_Las_limitaciones_de_los_modelos_recurrentes.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">